In [17]:
import pandas as pd

In [ ]:
import sys

sys.path.append("..")

from log_analyzer import LogAnalyzer

analyzer = LogAnalyzer(logs_dir="../logs")
df = analyzer.load_logs()

In [ ]:
df["extra"][0]

In [ ]:
pd.json_normalize(df["extra"], sep="_")

In [35]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_colwidth", None)

In [52]:
from datetime import datetime

df_clean = (
    df.assign(
        level_name=lambda x: x["level"].apply(lambda x: x["name"]),
        timestamp=lambda x: pd.to_datetime(x["time"].apply(lambda x: datetime.fromtimestamp(x["timestamp"]))),
        elapsed_seconds=lambda x: x["elapsed"].apply(lambda x: float(x["seconds"])),
        # id: process.id-thread.id
        id=lambda x: x["process"].apply(lambda x: x["id"]).astype(str)
        + "-"
        + x["thread"].apply(lambda x: x["id"]).astype(str),
        # extra = lambda x: x['extra'].apply(lambda x: json.loads(x['extra']) if x else None),
    )
    # concat extra to dataframe
    .join(pd.json_normalize(df["extra"], sep="_"), rsuffix="_extra")
)

In [ ]:
df_clean.head()

In [ ]:
(df_clean.loc[lambda x: x["level_name"] == "ERROR"])

In [ ]:
(
    df_clean.assign(
        ano=lambda x: x["timestamp"].dt.year,
        mes=lambda x: x["timestamp"].dt.month,
        ano_mes=lambda x: x["ano"].astype(str) + "-" + x["mes"].astype(str),
        ano_mes_dia=lambda x: x["ano_mes"] + "-" + x["timestamp"].dt.day.astype(str),
    )
    .loc[lambda x: ~(x["extra_rows"].isna())]
    .groupby(["ano_mes_dia", "extra_table"])
    .agg(
        sum_rows=("extra_rows", lambda x: x.sum()),
    )
)